# Advanced Problems with Solutions: Decorator Applications — Timing

Topic: timing decorators, benchmarking, recursion pitfalls, Fibonacci performance, decorator factories, memoization, metadata preservation, and decorator composition.

Each problem includes a full runnable solution.

## Problem 1 — Build a robust timing decorator

Write a decorator `timed` that:

- measures elapsed runtime using `perf_counter`,
- prints the function name and arguments,
- returns the original result,
- preserves metadata using `functools.wraps`,
- still prints timing information if the wrapped function raises an exception.

In [2]:
from functools import wraps
from time import perf_counter
import inspect

def timed(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        try:
            return fn(*args, **kwargs)
        finally:
            elapsed = perf_counter() - start
            args_part = [repr(arg) for arg in args]
            kwargs_part = [f"{key}={value!r}" for key, value in kwargs.items()]
            arguments = ", ".join(args_part + kwargs_part)
            print(f"{fn.__name__}({arguments}) took {elapsed:.6f}s")
    return wrapper

@timed
def add(a: int, b: int = 10) -> int:
    """Return a + b."""
    return a + b

assert add(2, b=3) == 5
assert add.__name__ == "add"
assert add.__doc__ == "Return a + b."
assert str(inspect.signature(add)) == "(a: int, b: int = 10) -> int"

print("All tests passed.")

add(2, b=3) took 0.000001s
All tests passed.


### Solution explanation

`perf_counter()` is preferred for measuring short durations. The `finally` block guarantees that timing information is printed whether the function succeeds or raises an exception. `@wraps(fn)` preserves metadata and allows `inspect.signature` to recover the original function signature.

## Problem 2 — Create a configurable timing decorator

Write a decorator factory `timed_factory(precision=6)`.

Example usage:

```python
@timed_factory(precision=4)
def fn(...):
    ...
```

The decorator should print elapsed time using the requested number of decimal places.

In [3]:
from functools import wraps
from time import perf_counter, sleep

def timed_factory(precision=6):
    if precision < 0:
        raise ValueError("precision must be non-negative")

    def decorator(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            start = perf_counter()
            try:
                return fn(*args, **kwargs)
            finally:
                elapsed = perf_counter() - start
                print(f"{fn.__name__} took {elapsed:.{precision}f}s")
        return wrapper
    return decorator

@timed_factory(precision=4)
def slow_identity(value):
    sleep(0.01)
    return value

assert slow_identity("Python") == "Python"
assert slow_identity.__name__ == "slow_identity"

print("All tests passed.")

slow_identity took 0.0105s
All tests passed.


### Solution explanation

A decorator with arguments requires two nested functions: the outer function receives configuration, and the inner decorator receives the function being decorated.

## Problem 3 — Explain and fix recursive timing noise

Decorating a recursive function directly causes every recursive call to be timed and printed.

Implement two versions of recursive Fibonacci:

1. A directly decorated recursive function.
2. A clean public wrapper that times only the total call.

In [4]:
from functools import wraps
from time import perf_counter

def timed(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = fn(*args, **kwargs)
        elapsed = perf_counter() - start
        print(f"{fn.__name__}{args} took {elapsed:.6f}s")
        return result
    return wrapper

@timed
def noisy_fib(n):
    if n <= 2:
        return 1
    return noisy_fib(n - 1) + noisy_fib(n - 2)

def raw_fib(n):
    if n <= 2:
        return 1
    return raw_fib(n - 1) + raw_fib(n - 2)

@timed
def clean_fib(n):
    return raw_fib(n)

assert noisy_fib(5) == 5
assert clean_fib(5) == 5

print("All tests passed.")

noisy_fib(2,) took 0.000001s
noisy_fib(1,) took 0.000001s
noisy_fib(3,) took 0.000129s
noisy_fib(2,) took 0.000000s
noisy_fib(4,) took 0.000152s
noisy_fib(2,) took 0.000000s
noisy_fib(1,) took 0.000000s
noisy_fib(3,) took 0.000021s
noisy_fib(5,) took 0.000196s
clean_fib(5,) took 0.000003s
All tests passed.


### Solution explanation

When a recursive function is decorated directly, each recursive call goes through the decorator. To time only the total computation, keep the recursive helper undecorated and decorate a non-recursive public wrapper.

## Problem 4 — Compare Fibonacci implementations

Implement Fibonacci using:

- naive recursion,
- a loop,
- `functools.reduce`.

Use a 1-based sequence:

```text
fib(1) = 1
fib(2) = 1
fib(3) = 2
```

In [5]:
from functools import reduce

def fib_recursive(n):
    if n < 1:
        raise ValueError("n must be >= 1")
    if n <= 2:
        return 1
    return fib_recursive(n - 1) + fib_recursive(n - 2)

def fib_loop(n):
    if n < 1:
        raise ValueError("n must be >= 1")
    if n <= 2:
        return 1
    a, b = 1, 1
    for _ in range(3, n + 1):
        a, b = b, a + b
    return b

def fib_reduce(n):
    if n < 1:
        raise ValueError("n must be >= 1")
    return reduce(lambda pair, _: (pair[1], pair[0] + pair[1]), range(n - 1), (0, 1))[1]

expected = {
    1: 1,
    2: 1,
    3: 2,
    4: 3,
    5: 5,
    6: 8,
    10: 55,
    20: 6765
}

for n, value in expected.items():
    assert fib_recursive(n) == value
    assert fib_loop(n) == value
    assert fib_reduce(n) == value

print("All tests passed.")

All tests passed.


### Solution explanation

The recursive implementation is the easiest to read but repeatedly recomputes the same values. The loop and reduce versions are linear-time algorithms. The loop version is usually clearer and often faster than the reduce version.

## Problem 5 — Write a repeated benchmark decorator

A single timing run can be noisy.

Write a decorator factory `benchmark(repeats=10)` that:

- runs the function multiple times,
- returns the result from the final run,
- prints average, minimum, and maximum elapsed time,
- preserves function metadata.

In [6]:
from functools import wraps
from time import perf_counter

def benchmark(repeats=10):
    if repeats < 1:
        raise ValueError("repeats must be at least 1")

    def decorator(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            timings = []
            result = None

            for _ in range(repeats):
                start = perf_counter()
                result = fn(*args, **kwargs)
                timings.append(perf_counter() - start)

            avg = sum(timings) / repeats
            print(
                f"{fn.__name__}: "
                f"avg={avg:.6f}s, "
                f"min={min(timings):.6f}s, "
                f"max={max(timings):.6f}s, "
                f"repeats={repeats}"
            )
            return result

        return wrapper
    return decorator

@benchmark(repeats=5)
def fib_loop_bench(n):
    a, b = 1, 1
    for _ in range(3, n + 1):
        a, b = b, a + b
    return b

assert fib_loop_bench(35) == 9227465
assert fib_loop_bench.__name__ == "fib_loop_bench"

print("All tests passed.")

fib_loop_bench: avg=0.000006s, min=0.000002s, max=0.000012s, repeats=5
All tests passed.


### Solution explanation

Repeated measurement is more informative than one measurement. Returning the final result keeps the decorated function usable as a normal function.

## Problem 6 — Add warmup runs to a benchmark decorator

Write `benchmark_with_warmup(repeats=10, warmups=3)`.

Warmup runs should execute the function but should not be included in the timing statistics.

In [7]:
from functools import wraps
from time import perf_counter
from statistics import mean, median

def benchmark_with_warmup(repeats=10, warmups=3):
    if repeats < 1:
        raise ValueError("repeats must be at least 1")
    if warmups < 0:
        raise ValueError("warmups must be non-negative")

    def decorator(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            for _ in range(warmups):
                fn(*args, **kwargs)

            timings = []
            result = None

            for _ in range(repeats):
                start = perf_counter()
                result = fn(*args, **kwargs)
                timings.append(perf_counter() - start)

            print(
                f"{fn.__name__}: "
                f"mean={mean(timings):.6f}s, "
                f"median={median(timings):.6f}s, "
                f"min={min(timings):.6f}s, "
                f"max={max(timings):.6f}s"
            )
            return result

        return wrapper
    return decorator

@benchmark_with_warmup(repeats=5, warmups=2)
def fib_reduce_bench(n):
    from functools import reduce
    return reduce(lambda pair, _: (pair[1], pair[0] + pair[1]), range(n - 1), (0, 1))[1]

assert fib_reduce_bench(35) == 9227465

print("All tests passed.")

fib_reduce_bench: mean=0.000006s, median=0.000006s, min=0.000005s, max=0.000006s
All tests passed.


### Solution explanation

Warmup runs can reduce startup effects and one-time overhead. The median is often useful because it is less sensitive to outliers than the mean.

## Problem 7 — Implement a memoization decorator

Write a decorator `memoize` for functions with hashable arguments.

Then use it to optimize recursive Fibonacci.

In [8]:
from functools import wraps

def memoize(fn):
    cache = {}

    @wraps(fn)
    def wrapper(*args, **kwargs):
        key = (args, tuple(sorted(kwargs.items())))
        if key not in cache:
            cache[key] = fn(*args, **kwargs)
        return cache[key]

    wrapper.cache = cache
    return wrapper

@memoize
def fib_memo(n):
    if n < 1:
        raise ValueError("n must be >= 1")
    if n <= 2:
        return 1
    return fib_memo(n - 1) + fib_memo(n - 2)

assert fib_memo(35) == 9227465
assert fib_memo(50) == 12586269025
assert fib_memo.__name__ == "fib_memo"

print("Cache size:", len(fib_memo.cache))
print("All tests passed.")

Cache size: 50
All tests passed.


### Solution explanation

Naive recursive Fibonacci is slow because it recomputes the same values many times. Memoization stores previous results and turns the recursive algorithm from exponential time into linear time for this problem.

## Problem 8 — Compose timing, caching, and counting decorators

Create three decorators:

- `count_calls`,
- `memoize`,
- `timed`.

Apply them to Fibonacci and observe how decorator order affects what is counted and timed.

In [9]:
from functools import wraps
from time import perf_counter

def count_calls(fn):
    count = 0

    @wraps(fn)
    def wrapper(*args, **kwargs):
        nonlocal count
        count += 1
        return fn(*args, **kwargs)

    wrapper.call_count = lambda: count
    return wrapper

def memoize(fn):
    cache = {}

    @wraps(fn)
    def wrapper(*args, **kwargs):
        key = (args, tuple(sorted(kwargs.items())))
        if key not in cache:
            cache[key] = fn(*args, **kwargs)
        return cache[key]

    wrapper.cache = cache
    return wrapper

def timed(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = fn(*args, **kwargs)
        elapsed = perf_counter() - start
        print(f"{fn.__name__}{args} took {elapsed:.6f}s")
        return result
    return wrapper

@timed
@count_calls
@memoize
def fib_a(n):
    if n <= 2:
        return 1
    return fib_a.__wrapped__.__wrapped__(n - 1) + fib_a.__wrapped__.__wrapped__(n - 2)

@count_calls
@timed
@memoize
def fib_b(n):
    if n <= 2:
        return 1
    return fib_b.__wrapped__.__wrapped__(n - 1) + fib_b.__wrapped__.__wrapped__(n - 2)

assert fib_a(10) == 55
assert fib_b(10) == 55

print("fib_b calls counted by outer counter:", fib_b.call_count())
print("All tests passed.")

fib_a(10,) took 0.000041s
fib_b(10,) took 0.000031s
fib_b calls counted by outer counter: 1
All tests passed.


### Solution explanation

Decorator order matters. The bottom decorator is applied first, and the top decorator becomes the outermost wrapper. In production code, prefer simpler recursive designs or `functools.lru_cache` over manually navigating `__wrapped__` chains.

## Problem 9 — Create a production-style benchmark report decorator

Write `benchmark_report(repeats=20)` that stores benchmark statistics on the decorated function as a dictionary called `.benchmark_stats`.

The dictionary should contain:

- `repeats`,
- `min`,
- `max`,
- `mean`,
- `median`.

In [10]:
from functools import wraps
from time import perf_counter
from statistics import mean, median

def benchmark_report(repeats=20):
    if repeats < 1:
        raise ValueError("repeats must be at least 1")

    def decorator(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            timings = []
            result = None

            for _ in range(repeats):
                start = perf_counter()
                result = fn(*args, **kwargs)
                timings.append(perf_counter() - start)

            wrapper.benchmark_stats = {
                "repeats": repeats,
                "min": min(timings),
                "max": max(timings),
                "mean": mean(timings),
                "median": median(timings)
            }

            return result

        wrapper.benchmark_stats = None
        return wrapper
    return decorator

@benchmark_report(repeats=10)
def fib_loop_report(n):
    a, b = 1, 1
    for _ in range(3, n + 1):
        a, b = b, a + b
    return b

assert fib_loop_report(100) == 354224848179261915075
assert fib_loop_report.benchmark_stats["repeats"] == 10
assert fib_loop_report.benchmark_stats["min"] <= fib_loop_report.benchmark_stats["max"]

print(fib_loop_report.benchmark_stats)
print("All tests passed.")

{'repeats': 10, 'min': 5.599111318588257e-06, 'max': 3.3099204301834106e-05, 'mean': 1.0629929602146148e-05, 'median': 7.400289177894592e-06}
All tests passed.


### Solution explanation

Storing benchmark statistics on the wrapper makes the data programmatically available instead of only printing it. This is useful for tests, reports, dashboards, and comparisons.

## Problem 10 — Choose readability over cleverness

The reduce-based Fibonacci implementation can be compressed into a one-liner, but the loop version is usually easier to understand.

Write both versions, verify they produce the same result, and explain which one you would choose in production code.

In [11]:
from functools import reduce

def fib_loop_readable(n):
    if n < 1:
        raise ValueError("n must be >= 1")
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

fib_reduce_oneliner = lambda n: reduce(
    lambda pair, _: (pair[1], pair[0] + pair[1]),
    range(n),
    (0, 1)
)[0]

for n in [1, 2, 3, 10, 35, 100]:
    assert fib_loop_readable(n) == fib_reduce_oneliner(n)

print("Both implementations are correct.")
print("In production, prefer the loop version unless there is a strong reason to use reduce.")

Both implementations are correct.
In production, prefer the loop version unless there is a strong reason to use reduce.


### Solution explanation

The loop version is direct, readable, and efficient. The reduce version is compact but harder to understand. Clever code is not automatically better code.